# Module E.1–E.4: Evaluating LLM Systems
**Part II — Applied LLM Engineering**

> Perplexity measured training loss. This module measures what users actually care about.

## Setup

We load a small causal LM (SmolLM2-135M-Instruct) and a sentence embedding model. Both run on CPU and fit in under 1 GB of RAM.

In [ ]:
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer
import numpy as np
import torch
import re
from collections import Counter

MODEL = "HuggingFaceTB/SmolLM2-135M-Instruct"
tok = AutoTokenizer.from_pretrained(MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
llm = AutoModelForCausalLM.from_pretrained(MODEL, torch_dtype=torch.float32)

def chat(messages, max_new_tokens=128, temperature=0.0):
    inputs = tok.apply_chat_template(
        messages, add_generation_prompt=True,
        return_tensors="pt", return_dict=True
    )
    do_sample = bool(temperature and temperature > 0)
    out = llm.generate(
        **inputs, max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=(temperature if do_sample else None),
        pad_token_id=tok.eos_token_id
    )
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

embed_model = SentenceTransformer("all-MiniLM-L6-v2")
print("Models loaded.")

## The Arborian Corpus

The same fictional world used in NB34/32. We repeat it here so this notebook is fully self-contained.

In [ ]:
CORPUS = [
    "The Arborian currency is called the Leaflet, made from compressed leaves.",
    "Arborians live in cities built entirely in the forest canopy, hundreds of meters above the ground.",
    "The Great Canopy Council governs Arboria and meets once per lunar cycle.",
    "Arborian diet consists mainly of photosynthetic supplements and rare forest fungi called Glowcaps.",
    "The traditional Arborian greeting is to press palms together and bow toward the oldest tree in view.",
    "Arborian children learn to climb before they learn to walk on flat surfaces.",
    "The annual Leaffall Festival celebrates the autumn shedding and lasts seven days.",
    "Transportation in Arboria relies on trained giant beetles called Carriers, each capable of bearing two passengers.",
    "Arborian medicine is based entirely on bark extracts and fungal compounds; they have no concept of synthetic drugs.",
    "The highest honor in Arboria is to be named Keeper of the Eldest Tree.",
]

# Small eval set: 5 question-answer pairs with ground truth and context
EVAL_SET = [
    {
        "question": "What is the Arborian currency?",
        "expected": "The Leaflet",
        "context": "The Arborian currency is called the Leaflet, made from compressed leaves.",
        "system_answer": "The Arborian currency is the Leaflet.",
    },
    {
        "question": "Where do Arborians live?",
        "expected": "In cities built in the forest canopy",
        "context": "Arborians live in cities built entirely in the forest canopy, hundreds of meters above the ground.",
        "system_answer": "Arborians reside in canopy cities high above the forest floor.",
    },
    {
        "question": "What do Arborians eat?",
        "expected": "Photosynthetic supplements and Glowcap fungi",
        "context": "Arborian diet consists mainly of photosynthetic supplements and rare forest fungi called Glowcaps.",
        "system_answer": "Their diet is made up of photosynthetic supplements and Glowcap fungi.",
    },
    {
        "question": "How do Arborians greet each other?",
        "expected": "Press palms together and bow toward the oldest tree",
        "context": "The traditional Arborian greeting is to press palms together and bow toward the oldest tree in view.",
        "system_answer": "They press their palms together and bow toward the nearest ancient tree.",
    },
    {
        "question": "What is the highest honor in Arboria?",
        "expected": "Being named Keeper of the Eldest Tree",
        "context": "The highest honor in Arboria is to be named Keeper of the Eldest Tree.",
        "system_answer": "The greatest distinction an Arborian can earn is the title of Keeper of the Eldest Tree.",
    },
]

print(f"Corpus: {len(CORPUS)} chunks")
print(f"Eval set: {len(EVAL_SET)} items")

## E.1 — Why Eval is Hard

For a classification problem, accuracy is clear: the label is either right or wrong. For LLMs the situation is completely different.

### Problem 1: No single ground truth

All three of these answers are correct for "What is the Arborian currency?":

- `"The Leaflet"`
- `"Arborians use Leaflets as currency"`
- `"Currency in Arboria is a compressed-leaf coin called the Leaflet"`

String equality gives 0 for all three against each other, yet they are semantically equivalent.

### Problem 2: Open-ended output space

A language model can produce infinitely many surface forms. Unlike image classification (1 of N classes), the output space is unbounded. Metrics that depend on exact overlap will always undercount quality.

### Problem 3: Evaluating quality at scale is itself expensive

Human annotation is the gold standard, but it costs money and time. If you ship a new model every week, you can't pay annotators every week. This is why we need automated metrics — but each one approximates human judgment imperfectly.

In [ ]:
# Concrete demonstration of Problem 1
question = "What is the Arborian currency?"
answers = [
    "The Leaflet",                                                     # terse, correct
    "Arborians use Leaflets as currency",                               # rephrased, correct
    "Currency in Arboria is a compressed-leaf coin called the Leaflet", # verbose, correct
    "Gold coins",                                                       # wrong
]
reference = "The Leaflet"

print(f"Reference: {reference!r}")
print()
for ans in answers:
    exact = int(ans.strip().lower() == reference.strip().lower())
    print(f"  Exact match={exact}  Answer: {ans!r}")

print()
print("String equality gives 0 for two perfectly correct paraphrases.")

## E.2 — Baseline Metrics: Exact Match and Token F1

These are the metrics used in the original SQuAD benchmark (2016). They remain the cheapest possible automated eval.

**Exact Match (EM)**: 1 if strings are identical after normalisation, 0 otherwise. Very strict.

**Token F1**: treat each string as a bag of tokens. Compute precision and recall over the token overlap. Rewards partial credit.

In [ ]:
def exact_match(predicted, expected):
    return int(predicted.strip().lower() == expected.strip().lower())

def token_f1(predicted, expected):
    pred_tokens = predicted.lower().split()
    exp_tokens = expected.lower().split()
    common = Counter(pred_tokens) & Counter(exp_tokens)
    num_common = sum(common.values())
    if num_common == 0:
        return 0.0
    precision = num_common / len(pred_tokens)
    recall = num_common / len(exp_tokens)
    return 2 * precision * recall / (precision + recall)

print(f"{'Question':<40} {'EM':>4} {'F1':>6}")
print("-" * 54)
for item in EVAL_SET:
    em = exact_match(item["system_answer"], item["expected"])
    f1 = token_f1(item["system_answer"], item["expected"])
    q_short = item["question"][:38]
    print(f"{q_short:<40} {em:>4} {f1:>6.3f}")

print()
print("EM is 0 for all — the system answers are rephrased, not verbatim.")
print("F1 is > 0 wherever there is token overlap, but it still penalises paraphrase.")

## E.3 — Semantic Similarity (BERTScore-style)

Token overlap is blind to synonyms and paraphrases. The key insight: encode both strings into a shared embedding space and measure cosine similarity. Two sentences that mean the same thing will cluster together even if they share no words.

This is the core idea behind BERTScore (Zhang et al., 2019), which uses contextual token embeddings. Here we use sentence-level embeddings for simplicity — same principle, less overhead.

In [ ]:
def semantic_similarity(predicted, expected, model):
    """Cosine similarity between sentence embeddings (normalized vectors -> dot product)."""
    vecs = model.encode([predicted, expected], normalize_embeddings=True)
    return float(vecs[0] @ vecs[1])

# First: show a paraphrase case that F1 misses
paraphrase_pairs = [
    ("The Leaflet", "Arborians use Leaflets as currency"),
    ("They live in canopy cities", "Arborian settlements are in the treetops"),
    ("Gold coins",  "The Leaflet"),  # wrong answer as a control
]

print("Paraphrase detection — where F1 fails and semantic similarity succeeds:")
print(f"{'Predicted':<42} {'Expected':<42} {'F1':>6} {'SemanticSim':>12}")
print("-" * 106)
for pred, exp in paraphrase_pairs:
    f1 = token_f1(pred, exp)
    sim = semantic_similarity(pred, exp, embed_model)
    print(f"{pred:<42} {exp:<42} {f1:>6.3f} {sim:>12.3f}")

In [ ]:
# Full comparison across the eval set
print(f"{'Question':<40} {'EM':>4} {'F1':>6} {'SemanticSim':>12}")
print("-" * 66)
for item in EVAL_SET:
    em  = exact_match(item["system_answer"], item["expected"])
    f1  = token_f1(item["system_answer"], item["expected"])
    sim = semantic_similarity(item["system_answer"], item["expected"], embed_model)
    q_short = item["question"][:38]
    print(f"{q_short:<40} {em:>4} {f1:>6.3f} {sim:>12.3f}")

print()
print("Semantic similarity scores the paraphrased answers much higher than F1 does.")

## E.4 — Faithfulness (Groundedness)

Semantic similarity tells us whether the answer matches a reference. But in a RAG system we sometimes don't have a reference — we have a **retrieved context**. The question becomes: *does the answer stick to what the context says, or does it invent facts?*

The metric is **faithfulness**: for each sentence in the answer, measure how closely it is supported by the context embedding. Low support = potential hallucination.

This is analogous to the faithfulness component in the RAGAS framework.

In [ ]:
def faithfulness_score(answer, context, model):
    """
    Mean similarity of each answer sentence to the context.
    High = answer is well-grounded in the context.
    Low  = answer contains claims not supported by context.
    """
    sentences = [s.strip() for s in re.split(r'[.!?]', answer) if s.strip()]
    if not sentences:
        return 1.0
    context_emb = model.encode([context], normalize_embeddings=True)[0]
    sent_embs = model.encode(sentences, normalize_embeddings=True)
    scores = sent_embs @ context_emb
    return float(np.mean(scores))

context = "The Arborian currency is called the Leaflet, made from compressed leaves."

grounded_answer = "The Arborian currency is the Leaflet, a coin made from pressed leaves."
hallucinated_answer = "The Arborian currency is the Solar, a gold coin minted by the Sun Council."

g_score = faithfulness_score(grounded_answer, context, embed_model)
h_score = faithfulness_score(hallucinated_answer, context, embed_model)

print("Context:")
print(f"  {context}")
print()
print(f"Grounded answer:     {grounded_answer!r}")
print(f"  Faithfulness score: {g_score:.3f}")
print()
print(f"Hallucinated answer: {hallucinated_answer!r}")
print(f"  Faithfulness score: {h_score:.3f}")
print()
print("Higher faithfulness = answer is better supported by the retrieved context.")

## Answer Relevance

An answer can be faithful (sticking to the context) and still be irrelevant (not actually addressing the question). **Answer relevance** measures whether the answer responds to what was asked.

In [ ]:
def answer_relevance(question, answer, model):
    """Cosine similarity between question and answer embeddings."""
    vecs = model.encode([question, answer], normalize_embeddings=True)
    return float(vecs[0] @ vecs[1])

question = "What is the Arborian currency?"
relevant_answer    = "The Arborian currency is called the Leaflet."
irrelevant_answer  = "Arborians live in cities built high in the forest canopy."
off_topic_answer   = "I don't know anything about that."

cases = [
    ("Relevant",   relevant_answer),
    ("Irrelevant", irrelevant_answer),
    ("Off-topic",  off_topic_answer),
]

print(f"Question: {question!r}")
print()
print(f"{'Label':<12} {'Score':>6}  Answer")
print("-" * 80)
for label, ans in cases:
    score = answer_relevance(question, ans, embed_model)
    print(f"{label:<12} {score:>6.3f}  {ans!r}")

## Context Relevance

In a RAG pipeline, retrieval can fail silently: the system returns chunks that score high in embedding space but are actually off-topic. **Context relevance** scores each retrieved chunk against the question so you can detect retrieval failures before they cause hallucinations.

This is important: a low context-relevance score means the model was *set up to hallucinate* by bad retrieval — not necessarily that the model itself is at fault.

In [ ]:
def context_relevance(question, chunks, model):
    """Score each retrieved chunk for relevance to the question."""
    q_emb = model.encode([question], normalize_embeddings=True)[0]
    chunk_embs = model.encode(chunks, normalize_embeddings=True)
    return (chunk_embs @ q_emb).tolist()

question = "What transport do Arborians use?"

# Simulate: retriever returned 4 chunks — 2 relevant, 2 not
retrieved_chunks = [
    "Transportation in Arboria relies on trained giant beetles called Carriers, each capable of bearing two passengers.",  # relevant
    "Arborian children learn to climb before they learn to walk on flat surfaces.",                                         # mildly relevant
    "The Arborian currency is called the Leaflet, made from compressed leaves.",                                           # irrelevant
    "The annual Leaffall Festival celebrates the autumn shedding and lasts seven days.",                                    # irrelevant
]

scores = context_relevance(question, retrieved_chunks, embed_model)

print(f"Question: {question!r}")
print()
print(f"{'Score':>6}  Chunk")
print("-" * 80)
for score, chunk in sorted(zip(scores, retrieved_chunks), reverse=True):
    tag = "[RELEVANT]" if score > 0.4 else "[WEAK]    "
    print(f"{score:>6.3f}  {tag}  {chunk[:70]}...")

print()
print("Low-scoring chunks should be filtered out before generation to reduce hallucination risk.")

## E.3 — LLM-as-Judge

Heuristic metrics are fast but shallow. An alternative: ask *another LLM* to evaluate the answer on multiple dimensions simultaneously — faithfulness, relevance, completeness — and output structured scores with reasoning.

This is the approach used in RAGAS, G-Eval, MT-Bench, and many production eval pipelines. The key tradeoff:

- **Pro**: can evaluate nuanced dimensions that embedding similarity cannot capture
- **Con**: the judge model introduces its own biases and errors; using the *same* weak model as judge and generator is circular

In production you would use a strong external judge (e.g. GPT-4 or Claude). Here we use SmolLM2-135M as the judge — results will be noisy, but the pattern is the same.

In [ ]:
def llm_judge(question, context, answer):
    """
    Ask the LLM to evaluate an answer on faithfulness, relevance, and completeness.
    Returns raw text (JSON if the model cooperates).

    NOTE: SmolLM2-135M is very small; JSON formatting will often be imperfect.
    In production use a larger judge model.
    """
    prompt = f"""You are an evaluator. Rate this answer on a scale of 1-5.

Question: {question}
Context: {context}
Answer: {answer}

Rate on: faithfulness (is it supported by context?), relevance (does it answer the question?), completeness (does it cover the key facts?).

Respond with JSON: {{"faithfulness": <1-5>, "relevance": <1-5>, "completeness": <1-5>, "reasoning": "<one sentence>"}}"""
    messages = [{"role": "user", "content": prompt}]
    return chat(messages, max_new_tokens=100)

item = EVAL_SET[0]
result = llm_judge(item["question"], item["context"], item["system_answer"])
print("LLM Judge output:")
print(result)
print()
print("[Note: small model may produce malformed JSON or inconsistent scores.]")

In [ ]:
# Run the judge on all eval items
print("Running LLM-as-judge on full eval set (may take ~1-2 minutes)...")
print()
for item in EVAL_SET:
    print(f"Q: {item['question']}")
    raw = llm_judge(item["question"], item["context"], item["system_answer"])
    print(f"Judge: {raw.strip()[:200]}")
    print()

## E.4 — Hallucination Detection

Faithfulness gives a single aggregate score. Hallucination detection goes further: it identifies *which specific sentences* in an answer are unsupported by the context. This is actionable — you can surface warnings to users or filter outputs before display.

The algorithm:
1. Split the answer into individual claims (sentences).
2. Embed each claim and the context.
3. Any claim whose similarity to the context falls below a threshold is flagged as potentially hallucinated.

In [ ]:
def detect_hallucination(answer, context, threshold=0.5):
    """
    Flag sentences in `answer` that are not well-supported by `context`.
    Returns a dict with:
      hallucinated: bool — True if any unsupported claims found
      unsupported_claims: list of flagged sentences
      scores: per-sentence similarity scores
    """
    sentences = [s.strip() for s in re.split(r'[.!?]', answer) if s.strip()]
    if not sentences:
        return {"hallucinated": False, "unsupported_claims": [], "scores": []}
    context_emb = embed_model.encode([context], normalize_embeddings=True)[0]
    sent_embs = embed_model.encode(sentences, normalize_embeddings=True)
    scores = sent_embs @ context_emb
    unsupported = [s for s, score in zip(sentences, scores) if score < threshold]
    return {
        "hallucinated": len(unsupported) > 0,
        "unsupported_claims": unsupported,
        "scores": scores.tolist(),
    }

context = "The Arborian currency is called the Leaflet, made from compressed leaves."

grounded_answer = (
    "The Arborian currency is the Leaflet. "
    "It is crafted from compressed leaves."
)
hallucinated_answer = (
    "The Arborian currency is the Leaflet. "
    "It is made of pure gold mined from the Deeproot Mountains. "
    "The Sun Council controls its minting."
)

for label, ans in [("Grounded", grounded_answer), ("Hallucinated", hallucinated_answer)]:
    result = detect_hallucination(ans, context)
    print(f"{label} answer:")
    print(f"  Text: {ans!r}")
    print(f"  Hallucinated: {result['hallucinated']}")
    for sent, score in zip(ans.split('. '), result['scores']):
        flag = " <-- FLAGGED" if score < 0.5 else ""
        print(f"    [{score:.3f}] {sent.strip()}{flag}")
    if result["unsupported_claims"]:
        print(f"  Unsupported claims: {result['unsupported_claims']}")
    print()

## Regression Testing

Individual metrics are interesting in isolation. The real value comes from wrapping them into an **eval suite that runs automatically** whenever the system changes — just like unit tests for code.

The pattern:
1. Maintain a **golden eval set** of question/expected-answer pairs (curated by humans).
2. On each model/prompt/retriever change, run the suite.
3. If `passed / total < threshold` (e.g. 0.8), fail the CI build.

This prevents regressions: a "fix" for one question that breaks three others will be caught immediately.

In [ ]:
def run_eval_suite(rag_fn, eval_set, thresholds=None):
    """
    Run the full eval suite against a RAG function.

    `rag_fn(question, context)` -> str (the system's answer)
    Each item in eval_set must have: question, expected, context.

    Returns a results dict with pass/fail summary and per-item details.
    """
    if thresholds is None:
        thresholds = {"semantic_sim": 0.6, "faithfulness": 0.5}

    results = []
    passed = 0

    for item in eval_set:
        answer = rag_fn(item["question"], item["context"])
        sem_sim = semantic_similarity(answer, item["expected"], embed_model)
        faith   = faithfulness_score(answer, item["context"], embed_model)
        rel     = answer_relevance(item["question"], answer, embed_model)
        ok = sem_sim >= thresholds["semantic_sim"] and faith >= thresholds["faithfulness"]

        results.append({
            "question":     item["question"],
            "expected":     item["expected"],
            "answer":       answer,
            "semantic_sim": round(sem_sim, 3),
            "faithfulness": round(faith, 3),
            "relevance":    round(rel, 3),
            "pass":         ok,
        })
        if ok:
            passed += 1

    return {
        "passed": passed,
        "total":  len(eval_set),
        "pass_rate": passed / len(eval_set),
        "results": results,
    }

print("run_eval_suite defined.")

## A Complete Eval Run

Now we put everything together. We define a minimal RAG system (retrieval by embedding similarity + generation via `chat()`), run it over all 5 eval items, and print a summary table with all metrics.

In [ ]:
# Pre-embed the corpus once
corpus_embs = embed_model.encode(CORPUS, normalize_embeddings=True)

def retrieve(question, k=2):
    """Return the top-k corpus chunks most similar to the question."""
    q_emb = embed_model.encode([question], normalize_embeddings=True)[0]
    scores = corpus_embs @ q_emb
    top_k  = np.argsort(scores)[::-1][:k]
    return [CORPUS[i] for i in top_k]

def rag_answer(question, context=None):
    """
    Simple RAG: retrieve relevant chunks, build a prompt, generate an answer.
    If context is supplied (for eval purposes) it is used directly.
    """
    if context is None:
        chunks = retrieve(question)
        context = " ".join(chunks)
    messages = [
        {
            "role": "system",
            "content": "You are a helpful assistant. Answer ONLY using the provided context. Be concise.",
        },
        {
            "role": "user",
            "content": f"Context: {context}\n\nQuestion: {question}",
        },
    ]
    return chat(messages, max_new_tokens=80)

print("RAG system defined. Running eval suite...")
print()

suite_output = run_eval_suite(rag_answer, EVAL_SET)

# Print a summary table
header = f"{'Question':<42} {'SemanticSim':>12} {'Faithful':>9} {'Relevance':>10} {'Pass':>5}"
print(header)
print("-" * len(header))
for r in suite_output["results"]:
    q_short = r["question"][:40]
    ok      = "PASS" if r["pass"] else "FAIL"
    print(f"{q_short:<42} {r['semantic_sim']:>12.3f} {r['faithfulness']:>9.3f} {r['relevance']:>10.3f} {ok:>5}")

print()
print(f"Result: {suite_output['passed']}/{suite_output['total']} passed  "
      f"({suite_output['pass_rate']*100:.0f}%)")
pass_rate = suite_output['pass_rate']
if pass_rate >= 0.8:
    print("CI status: BUILD PASS")
else:
    print("CI status: BUILD FAIL  (pass rate below 80% threshold)")

In [ ]:
# Print per-item answer details
print("Per-item detail:\n")
for r in suite_output["results"]:
    print(f"Q: {r['question']}")
    print(f"   Expected: {r['expected']}")
    print(f"   Got:      {r['answer'].strip()[:120]}")
    print(f"   Metrics:  semantic_sim={r['semantic_sim']:.3f}  faithfulness={r['faithfulness']:.3f}  relevance={r['relevance']:.3f}")
    print()

## What You've Built

Here is the full pipeline in one place:

```
Query
  └─> Retrieval
        └─> [Context Relevance check]        <- catches bad retrieval
        └─> Generation
              └─> [Faithfulness check]       <- catches hallucination
              └─> [Answer Relevance check]   <- catches tangential answers
              └─> [Semantic Similarity]      <- compares to golden answers
              └─> [LLM-as-Judge]             <- multi-dimension qualitative score
```

Each check can be a gate (block output below threshold), a monitor (log for analysis), or a CI signal (fail build if aggregate drops). Which role each plays is a product decision, not a technical one.

## Try It Yourself

Three exercises to consolidate what you've built:

### Task A — Citation Accuracy Metric

Add a `citation_accuracy` metric that checks whether a cited chunk index (e.g. `[1]`, `[2]`) appears in a list of valid retrieved chunk indices. If the answer cites `[3]` but only chunks 0 and 1 were retrieved, that is a hallucinated citation.

```python
def citation_accuracy(answer, valid_chunk_indices):
    """
    Find all [N] citations in answer.
    Return fraction of cited indices that are in valid_chunk_indices.
    If no citations, return 1.0 (vacuously accurate).
    """
    # YOUR CODE HERE
    cited = [int(m) for m in re.findall(r'\[(\d+)\]', answer)]
    if not cited:
        return 1.0
    valid = sum(1 for c in cited if c in valid_chunk_indices)
    return valid / len(cited)

# Test it:
print(citation_accuracy("Arborians use the Leaflet [0] for all trade [5].", {0, 1}))  # 0.5
print(citation_accuracy("Arborians use the Leaflet [0].", {0, 1}))                   # 1.0
print(citation_accuracy("Arborians use the Leaflet.", {0, 1}))                       # 1.0 (no citations)
```

### Task B — Build Your Own Golden Set

Pick 3 corpus chunks you haven't used. For each, write:
1. A natural question whose answer is in that chunk.
2. A short, canonical expected answer.
3. A paraphrased expected answer.

Run `semantic_similarity` between the paraphrase and the canonical answer. It should be > 0.7. If not, rewrite until it is.

### Task C — Add Latency to Eval Results

Modify `run_eval_suite` to record how long generation takes for each item. Add a `"latency_ms"` key to each result dict. Then add a `"mean_latency_ms"` to the suite output.

```python
import time

# Inside the loop:
t0 = time.perf_counter()
answer = rag_fn(item["question"], item["context"])
latency_ms = (time.perf_counter() - t0) * 1000
```

Latency is a first-class eval dimension in production systems — a model that is 5% more accurate but 3x slower may not be the right trade-off.

In [ ]:
# Task A scaffold — fill in the function and run the assertions
def citation_accuracy(answer, valid_chunk_indices):
    # YOUR CODE HERE
    pass

# Uncomment to test once implemented:
# assert citation_accuracy("The Leaflet [0] is used everywhere [5].", {0, 1}) == 0.5
# assert citation_accuracy("The Leaflet [0].", {0, 1}) == 1.0
# assert citation_accuracy("The Leaflet.", {0, 1}) == 1.0
# print("Task A: citation_accuracy OK")

In [ ]:
# Task B — build your golden set
MY_GOLDEN_SET = [
    # {
    #     "question": "...",
    #     "expected": "...",
    #     "context": CORPUS[?],
    # },
]

# Once you fill it in, run:
# for item in MY_GOLDEN_SET:
#     print(item["question"])
#     print("  Semantic sim (paraphrase vs canonical):",
#           semantic_similarity(item["expected"], item["context"], embed_model))

print("Fill in MY_GOLDEN_SET above with at least 3 items.")

In [ ]:
# Task C — run_eval_suite with latency
import time

def run_eval_suite_with_latency(rag_fn, eval_set, thresholds=None):
    if thresholds is None:
        thresholds = {"semantic_sim": 0.6, "faithfulness": 0.5}

    results = []
    passed = 0

    for item in eval_set:
        # YOUR CODE: wrap rag_fn call with time.perf_counter()
        answer = rag_fn(item["question"], item["context"])
        latency_ms = 0.0  # replace with measured latency

        sem_sim = semantic_similarity(answer, item["expected"], embed_model)
        faith   = faithfulness_score(answer, item["context"], embed_model)
        rel     = answer_relevance(item["question"], answer, embed_model)
        ok = sem_sim >= thresholds["semantic_sim"] and faith >= thresholds["faithfulness"]

        results.append({
            "question":     item["question"],
            "answer":       answer,
            "semantic_sim": round(sem_sim, 3),
            "faithfulness": round(faith, 3),
            "relevance":    round(rel, 3),
            "latency_ms":   round(latency_ms, 1),  # add measurement above
            "pass":         ok,
        })
        if ok:
            passed += 1

    latencies = [r["latency_ms"] for r in results]
    return {
        "passed": passed,
        "total":  len(eval_set),
        "pass_rate":      passed / len(eval_set),
        "mean_latency_ms": round(sum(latencies) / len(latencies), 1),
        "results": results,
    }

print("Task C: add timing to the two lines marked above, then run:")
print("  out = run_eval_suite_with_latency(rag_answer, EVAL_SET)")
print("  print(out['mean_latency_ms'], 'ms per item')")